In [16]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, TextContentBlock, SystemMessage, AIMessage

load_dotenv(override=True)

llm = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    profile={
      "max_input_tokens":1_280_000
    },
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

In [19]:
messages = [
    SystemMessage(content_blocks=[
        TextContentBlock(type="text",text="你是一个非常友好的ai助手")
    ]),
    HumanMessage(content="你好呀，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴的太早了"),
    HumanMessage("呵呵，你什么意思？")
]

In [6]:
print(llm.profile)

{}


## 1 SummarizationMiddleware中间件

### 举例1： 测试trigger keep参数

In [20]:
agent = create_agent(
    model=llm,
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001),
            ],
            keep=("messages",2)
        )
    ]
)

resp = agent.invoke({
    "messages":messages
})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户（老王）主动打招呼并询问 AI 助手的身份，希望建立初步认识。

## SUMMARY
- 用户自称"老王"，向 AI 打招呼并询问 AI 的身份。
- AI 回复自称"小王"，完成了自我介绍。
- 用户表示很高兴认识"小王"。
- 这是一次简单的寒暄对话，双方建立了初步的称呼和关系（老王 ↔ 小王）。

## ARTIFACTS
None

## NEXT STEPS
None（对话已完成初步寒暄，等待用户提出后续需求）
================================== Ai Message ==================================

你高兴的太早了
================================ Human Message =================================

呵呵，你什么意思？
================================== Ai Message ==================================

哈哈，开个玩笑，别介意！老王你好，我是小王，有什么需要帮忙的尽管说。


### 举例2: summary_prompt

In [21]:
agent = create_agent(
    model=llm,
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001),
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息如下\n{messages}"

        )
    ]
)

resp = agent.invoke({
    "messages":messages
})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

好的，以下是对这段历史消息的摘要：

**摘要：**
用户老王向 AI 打招呼并自我介绍，询问 AI 身份。AI 自称“小王”，表示很高兴认识老王。双方完成初次问候与认识。
================================== Ai Message ==================================

你高兴的太早了
================================ Human Message =================================

呵呵，你什么意思？
================================== Ai Message ==================================

哈哈，开个玩笑，别介意。我的意思是，咱们才刚认识，后面说不定还有好多有意思的事儿呢，高兴的日子还在后头。有什么我能帮你的，尽管说。
